# Top anomalies to explain in parallel

In [33]:
import os
import numpy as np
import pandas as pd
from anomaly.utils import specobjid_to_idx

from sdss.metadata import MetaData

meta = MetaData()

# Custom functions

# Config

## Residuals

In [16]:
se_family = ["mse", "mse_97", "mse_filter_250", "mse_filter_250_97"]
rse_family = [f"{col}_rel" for col in se_family]
se_and_rse_family = se_family + rse_family

## Directories

In [32]:
phd_dir = "/home/elom/phd"
data_dir = f"{phd_dir}/code"
spectra_dir = f"{data_dir}/spectra"
scores_dir = f"{data_dir}/scores"
explanations_dir = f"{data_dir}/explanations"
latent_dir = f"{data_dir}/latent"

## Data

In [10]:
wave = np.load(f"{spectra_dir}/wave_spectra_imputed.npy")
wave_nm = wave * 0.1

spectra = np.load(f"{spectra_dir}/spectra_imputed.npy", mmap_mode="r")

final_meta_df = pd.read_csv(
    f"{spectra_dir}/final_spec_n_z_warning_drop.csv.gz",
    index_col="specobjid",
)

idx_id_spec = np.load(f"{spectra_dir}/ids_imputing.npy", mmap_mode="r")

## Scores

In [12]:
bin_id = "bin_03"
scores_df = pd.read_csv(
    f"{scores_dir}/{bin_id}/scores_{bin_id}.csv.gz",
    index_col="specobjid",
)

In [14]:
scores_df.columns

Index(['mse_rel', 'mse_filter_250_97_rel', 'mse_95', 'mse_97_rel', 'mse',
       'mse_97', 'mse_filter_250_95_rel', 'mse_filter_300_95', 'mse_95_rel',
       'mse_filter_300', 'mse_filter_250_97', 'mse_filter_300_95_rel',
       'mse_filter_250', 'mse_filter_300_97', 'mse_filter_250_95',
       'mse_filter_300_97_rel', 'mse_filter_300_rel', 'mse_filter_250_rel'],
      dtype='object')

# Top anomalies for residual based scores

In [34]:
objid_top_anomalies = {}
spec_top_anomalies = {}
bin_id = "bin_03"

for score in se_and_rse_family:

    save_to = f"{explanations_dir}/{bin_id}/{score}"
    os.makedirs(save_to, exist_ok=True)

    top_1_threshold = scores_df[score].quantile(0.99)
    top_1_mask = scores_df[score] >= top_1_threshold
    n_top_1 = top_1_mask.sum()

    print(f"Save {n_top_1} anomalies for {score} at:\n{save_to}")

    objids = scores_df[top_1_mask].index.to_numpy()
    objid_top_anomalies[score] = objids
    indices = [specobjid_to_idx(specid, idx_id_spec) for specid in objids]
    spec_top_anomalies[score] = spectra[indices]
    # save spectra and objids for top anomalies
    np.save(f"{save_to}/specobjids_anomalies.npy", objids)
    np.save(f"{save_to}/spectra_anomalies.npy", spectra[indices])

Save 1819 anomalies for mse at:
/home/elom/phd/code/explanations/bin_03/mse
Save 1819 anomalies for mse_97 at:
/home/elom/phd/code/explanations/bin_03/mse_97
Save 1819 anomalies for mse_filter_250 at:
/home/elom/phd/code/explanations/bin_03/mse_filter_250
Save 1819 anomalies for mse_filter_250_97 at:
/home/elom/phd/code/explanations/bin_03/mse_filter_250_97
Save 1819 anomalies for mse_rel at:
/home/elom/phd/code/explanations/bin_03/mse_rel
Save 1819 anomalies for mse_97_rel at:
/home/elom/phd/code/explanations/bin_03/mse_97_rel
Save 1819 anomalies for mse_filter_250_rel at:
/home/elom/phd/code/explanations/bin_03/mse_filter_250_rel
Save 1819 anomalies for mse_filter_250_97_rel at:
/home/elom/phd/code/explanations/bin_03/mse_filter_250_97_rel


In [35]:
spec_top_anomalies[score].shape

(1819, 3773)